# M3L3 E13 — Sistema completo con estado (Resolution)
### Módulo 3 · Lecture 3 · Sistemas Multiagente

## ¿Qué vas a aprender hoy?
- agregar sesión, guardrails y conversación multi-turno.
- Conectar el concepto con M3L2.
- Leer código pequeño con explicación previa.
- Interpretar resultados y checks.


## ¿Qué necesitás saber antes?

Venís de M3L2 con LangChain, LCEL, PromptTemplate, RAG con FAISS y memoria conversacional. En M3L3 usamos esas piezas para coordinar varios agentes.

> **Sistema multiagente:** arquitectura donde varias unidades especializadas colaboran bajo una política de coordinación.


## Instalación e imports

En un notebook productivo podrías instalar `langchain`, `langchain-openai` y `faiss-cpu`. Aquí usamos Python estándar para que el foco sea el diseño multiagente y no la API key.


In [ ]:
from typing import Callable, TypedDict, Literal
from dataclasses import dataclass, field
import json

print("Setup listo: usamos Python estándar para que el notebook pueda correr sin API key.")


## Sección 1 — Estado conversacional

La sesión guarda historial, contexto y agentes visitados. Así una pregunta de seguimiento puede usar el tema anterior.

> **Guardrail:** regla que limita el sistema para evitar loops, exceso de handoffs o respuestas inseguras.


Creamos `SessionState` con dataclass. Cada sesión tiene sus propias listas y diccionarios.


In [ ]:
@dataclass
class SessionState:
    visited_agents: list[str] = field(default_factory=list)
    turn_count: int = 0
    context: dict = field(default_factory=dict)
    history: list[dict] = field(default_factory=list)


## Sección 2 — Guardrails, fallback y handle_query

`handle_query(query, state)` devuelve `(response, state)`. Ese contrato hace explícito cómo fluye la memoria entre turnos.


In [ ]:
def check_guardrails(state: SessionState) -> str | None:
    if state.turn_count >= 5: return "max_turns"
    if len(state.visited_agents) > 4: return "max_handoffs"
    return None

def fallback_response(reason: str) -> str:
    return {"max_turns":"Límite de turnos de la demo.", "max_handoffs":"Demasiadas transferencias; escalar a humano.", "unknown":"No pude identificar HR, Tech o Billing."}.get(reason, "No puedo resolver esto con confianza.")

def detect_agent(query: str, state: SessionState) -> str:
    text = query.lower()
    if "vacaciones" in text or "seguro" in text: state.context["last_topic"] = "hr"; return "hr"
    if "vpn" in text or "contraseña" in text: state.context["last_topic"] = "tech"; return "tech"
    if "factura" in text or "reembolso" in text: state.context["last_topic"] = "billing"; return "billing"
    if "cómo" in text or "como" in text or "mientras" in text: return state.context.get("last_topic", "unknown")
    return "unknown"

def answer_with_agent(agent: str) -> str:
    return {"hr":"HRAgent: solicitá vacaciones en PeopleOps.", "tech":"TechAgent: reiniciá VPN y validá MFA.", "billing":"BillingAgent: cargá comprobante y centro de costo."}.get(agent, fallback_response("unknown"))

def handle_query(query: str, state: SessionState) -> tuple[str, SessionState]:
    guardrail = check_guardrails(state)
    if guardrail: return fallback_response(guardrail), state
    state.turn_count += 1
    agent = detect_agent(query, state)
    if agent != "unknown": state.visited_agents.append(agent)
    response = answer_with_agent(agent)
    state.history.append({"query": query, "agent": agent, "response": response})
    return response, state


## Sección 3 — Conversación multi-turno

Simulamos cinco turnos. Observá `last_topic`, `turn_count` y `visited_agents`.


In [ ]:
state = SessionState()
for q in ["¿Cuántos días de vacaciones tengo?", "¿Y cómo los solicito?", "Mi VPN no conecta", "¿Puedo hacer algo mientras?", "Gracias"]:
    response, state = handle_query(q, state)
    print("Usuario:", q)
    print("Bot:", response)
    print({"turns": state.turn_count, "last_topic": state.context.get("last_topic"), "visited": state.visited_agents})
    print("---")
edge = SessionState(turn_count=5)
print(handle_query("vpn", edge)[0])


## Checks automáticos

Los checks verifican el contrato mínimo del ejercicio. En Starter pueden fallar hasta completar los TODOs; en Resolution deben pasar.


In [ ]:
def run_checks():
    s = SessionState()
    response, s = handle_query("vacaciones", s)
    assert s.turn_count == 1
    assert s.history
    assert isinstance(response, str)
    print("Checks E13 OK")
run_checks()


## ¿Qué aprendiste hoy?

- Agregar sesión, guardrails y conversación multi-turno.
- Separar responsabilidades vuelve el sistema más auditable.
- Los contratos explícitos hacen que el orquestador dependa menos de texto libre.

## Próximo ejercicio

Continuá con el siguiente notebook de M3L3 para agregar una pieza más de coordinación multiagente.
